# C08 Top-3 reseed

Re-train the three finalist challenger recipes across multiple random seeds, then save per-run metrics for stability analysis.


In [ ]:
import json, random, numpy as np, pandas as pd, torch, joblib
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset


In [ ]:
SEEDS = [7, 13, 21]
BASE_MODEL = "bert-base-uncased"
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
MIN_SUPPORT = 30

RUN_SPECS = [
    {
        "base_model_name": "label_smoothing_eps01_2ep",
        "method": "Label Smoothing + sqrt_rw",
        "recipe": "label_smoothing",
        "epochs": 2,
        "smoothing": 0.10,
    },
    {
        "base_model_name": "rdrop_alpha10_2ep",
        "method": "R-Drop + sqrt_rw",
        "recipe": "rdrop",
        "epochs": 2,
        "kl_alpha": 1.0,
    },
    {
        "base_model_name": "weighted_sampler_citypow15_2ep",
        "method": "Weighted Sampler (city_group) + Plain CE",
        "recipe": "weighted_sampler",
        "epochs": 2,
        "sampling_power": 1.5,
    },
]


In [ ]:
CWD = Path.cwd()
NOTEBOOKS_DIR = CWD.parent if CWD.name == "challengers" else CWD
PROJECT_ROOT = NOTEBOOKS_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "notebooks" / "models" / "challengers"
FIGURES_DIR = PROJECT_ROOT / "figures" / "challengers"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR exists: {(DATA_DIR / 'train.csv').exists()}")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"FIGURES_DIR: {FIGURES_DIR}")
print(f"Seeds: {SEEDS}")


In [ ]:
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_val = pd.read_csv(DATA_DIR / "val.csv")
df_test = pd.read_csv(DATA_DIR / "test.csv")

mapping = pd.read_csv(DATA_DIR / "label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))
for df in [df_train, df_val, df_test]:
    df["supercategory"] = df["label"].map(label_to_supercat)

le = LabelEncoder()
df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])
num_labels = len(le.classes_)

city_counts = df_train["city_group"].value_counts()
sqrt_rw_raw = 1.0 / np.sqrt(city_counts)
sqrt_rw_map = (sqrt_rw_raw / sqrt_rw_raw.mean()).to_dict()
df_train["sample_weight"] = df_train["city_group"].map(sqrt_rw_map).astype(float)

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print(f"Labels: {num_labels}, City groups: {len(city_counts)}")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch["resume_text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

train_weighted_ds = Dataset.from_pandas(df_train[["resume_text", "y", "sample_weight"]]).map(tokenize, batched=True)
train_plain_ds = Dataset.from_pandas(df_train[["resume_text", "y"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(df_val[["resume_text", "y"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(df_test[["resume_text", "y"]]).map(tokenize, batched=True)

train_weighted_ds = train_weighted_ds.rename_column("y", "labels")
train_plain_ds = train_plain_ds.rename_column("y", "labels")
val_ds = val_ds.rename_column("y", "labels")
test_ds = test_ds.rename_column("y", "labels")

train_weighted_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "sample_weight"])
train_plain_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_metrics(prediction_output):
    preds = np.argmax(prediction_output.predictions, axis=1)
    return {
        "accuracy": accuracy_score(prediction_output.label_ids, preds),
        "macro_f1": f1_score(prediction_output.label_ids, preds, average="macro"),
    }


def ovr_rates(df, group_col, num_classes):
    groups = sorted(df[group_col].dropna().unique())
    tpr = np.zeros((len(groups), num_classes))
    support = np.zeros((len(groups), num_classes))
    for gi, group_name in enumerate(groups):
        dg = df[df[group_col] == group_name]
        yt, yp = dg["y_true"].values, dg["y_pred"].values
        for c in range(num_classes):
            positive_mask = yt == c
            tp = np.sum((yp == c) & positive_mask)
            fn = np.sum((yp != c) & positive_mask)
            support[gi, c] = positive_mask.sum()
            tpr[gi, c] = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    return tpr, support


def robust_gaps(tpr, support, min_support=30):
    gaps = []
    for c in range(tpr.shape[1]):
        col = tpr[support[:, c] >= min_support, c]
        col = col[~np.isnan(col)]
        gaps.append(col.max() - col.min() if len(col) >= 2 else np.nan)
    gaps = np.array(gaps)
    valid = gaps[~np.isnan(gaps)]
    return (valid.max() if len(valid) else np.nan, valid.mean() if len(valid) else np.nan)


def make_sampler_weights(groups, power):
    counts = groups.value_counts()
    raw = 1.0 / np.power(counts.astype(float), power)
    normalized = raw / raw.mean()
    weights = groups.map(normalized).astype(float).to_numpy()
    return weights, normalized.to_dict()


class LabelSmoothingTrainer(Trainer):
    def __init__(self, *args, smoothing=0.1, num_classes=9, **kwargs):
        super().__init__(*args, **kwargs)
        self.smoothing = smoothing
        self.num_classes = num_classes

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        sample_weight = inputs.pop("sample_weight", None)
        labels = inputs["labels"]
        outputs = model(**inputs)
        logits = outputs.logits
        log_probs = F.log_softmax(logits, dim=-1)
        smooth_targets = torch.full_like(log_probs, self.smoothing / (self.num_classes - 1))
        smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - self.smoothing)
        loss_per_sample = -(smooth_targets * log_probs).sum(dim=-1)
        if sample_weight is not None:
            loss_per_sample = loss_per_sample * sample_weight.to(loss_per_sample.dtype)
        loss = loss_per_sample.mean()
        return (loss, outputs) if return_outputs else loss


def symmetric_kl(logits_p, logits_q):
    p_log = F.log_softmax(logits_p, dim=-1)
    q_log = F.log_softmax(logits_q, dim=-1)
    p = p_log.exp()
    q = q_log.exp()
    kl_pq = F.kl_div(p_log, q, reduction="none").sum(dim=-1)
    kl_qp = F.kl_div(q_log, p, reduction="none").sum(dim=-1)
    return 0.5 * (kl_pq + kl_qp)


class RDropTrainer(Trainer):
    def __init__(self, *args, kl_alpha=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.kl_alpha = kl_alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        sample_weight = inputs.pop("sample_weight", None)
        labels = inputs["labels"]
        outputs_a = model(**inputs)
        outputs_b = model(**inputs)
        logits_a = outputs_a.logits
        logits_b = outputs_b.logits
        ce_a = F.cross_entropy(logits_a, labels, reduction="none")
        ce_b = F.cross_entropy(logits_b, labels, reduction="none")
        ce_loss = 0.5 * (ce_a + ce_b)
        kl_loss = symmetric_kl(logits_a, logits_b)
        total = ce_loss + self.kl_alpha * kl_loss
        if sample_weight is not None:
            total = total * sample_weight.to(total.dtype)
        loss = total.mean()
        return (loss, outputs_a) if return_outputs else loss


class WeightedSamplerTrainer(Trainer):
    def __init__(self, *args, sampler_weights, **kwargs):
        super().__init__(*args, **kwargs)
        self.sampler_weights = torch.as_tensor(sampler_weights, dtype=torch.double)

    def get_train_dataloader(self):
        sampler = WeightedRandomSampler(
            weights=self.sampler_weights,
            num_samples=len(self.train_dataset),
            replacement=True,
        )
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )


In [ ]:
def make_training_args(checkpoint_dir, seed, epochs):
    return TrainingArguments(
        output_dir=str(checkpoint_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=epochs,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        remove_unused_columns=False,
        report_to="none",
        seed=seed,
        data_seed=seed,
    )


summary_rows = []

for spec in RUN_SPECS:
    for seed in SEEDS:
        set_seed(seed)
        model_name = f"{spec['base_model_name']}_seed{seed}"
        save_dir = MODELS_DIR / model_name
        checkpoint_dir = save_dir / "checkpoints"

        print()
        print("=" * 80)
        print(f"Training {model_name}")

        model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=num_labels)
        args = make_training_args(checkpoint_dir, seed, spec["epochs"])

        if spec["recipe"] == "label_smoothing":
            trainer = LabelSmoothingTrainer(
                model=model,
                args=args,
                train_dataset=train_weighted_ds,
                eval_dataset=val_ds,
                compute_metrics=compute_metrics,
                smoothing=spec["smoothing"],
                num_classes=num_labels,
            )
            extra_config = {"smoothing": spec["smoothing"]}
        elif spec["recipe"] == "rdrop":
            trainer = RDropTrainer(
                model=model,
                args=args,
                train_dataset=train_weighted_ds,
                eval_dataset=val_ds,
                compute_metrics=compute_metrics,
                kl_alpha=spec["kl_alpha"],
            )
            extra_config = {"kl_alpha": spec["kl_alpha"]}
        elif spec["recipe"] == "weighted_sampler":
            sampler_weights, sampler_map = make_sampler_weights(df_train["city_group"], spec["sampling_power"])
            trainer = WeightedSamplerTrainer(
                model=model,
                args=args,
                train_dataset=train_plain_ds,
                eval_dataset=val_ds,
                compute_metrics=compute_metrics,
                sampler_weights=sampler_weights,
            )
            extra_config = {
                "sampling_power": spec["sampling_power"],
                "sampler_weights_by_city": sampler_map,
            }
        else:
            raise ValueError(f"Unknown recipe: {spec['recipe']}")

        trainer.train()
        pred_output = trainer.predict(test_ds)
        y_true = pred_output.label_ids
        y_pred = np.argmax(pred_output.predictions, axis=1)

        acc = float(accuracy_score(y_true, y_pred))
        macro_f1 = float(f1_score(y_true, y_pred, average="macro"))

        df_eval = df_test.copy()
        df_eval["y_true"] = y_true
        df_eval["y_pred"] = y_pred
        tpr, support = ovr_rates(df_eval, "city_group", num_labels)
        worst_gap, macro_gap = robust_gaps(tpr, support, min_support=MIN_SUPPORT)

        save_dir.mkdir(parents=True, exist_ok=True)
        trainer.model.save_pretrained(save_dir, safe_serialization=True)
        tokenizer.save_pretrained(save_dir)
        joblib.dump(le, save_dir / "label_encoder.joblib")

        config = {
            "method": spec["method"],
            "recipe": spec["recipe"],
            "base_model_name": spec["base_model_name"],
            "seed": seed,
            "epochs": spec["epochs"],
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "accuracy": acc,
            "macro_f1": macro_f1,
            "tpr_gap_worst_robust": float(worst_gap),
            "tpr_gap_macro_robust": float(macro_gap),
        }
        config.update(extra_config)
        with open(save_dir / "training_config.json", "w") as f:
            json.dump(config, f, indent=2)

        summary_rows.append({
            "model_name": model_name,
            "base_model_name": spec["base_model_name"],
            "method": spec["method"],
            "recipe": spec["recipe"],
            "seed": seed,
            "epochs": spec["epochs"],
            "accuracy": acc,
            "macro_f1": macro_f1,
            "worst_gap": float(worst_gap),
            "macro_gap": float(macro_gap),
            "model_dir": str(save_dir.relative_to(PROJECT_ROOT)),
        })

summary_df = pd.DataFrame(summary_rows).sort_values(["base_model_name", "seed"]).reset_index(drop=True)
summary_path = FIGURES_DIR / "c08_top3_reseed_runs.csv"
summary_df.to_csv(summary_path, index=False)
summary_df


In [ ]:
print(f"Saved reseed runs to: {summary_path}")
summary_df.groupby("base_model_name")[["accuracy", "macro_f1", "worst_gap", "macro_gap"]].agg(["mean", "std", "min", "max"])
